# Korean BERT Sentiment Analysis with NSMC

Kaggle Notebook에서 바로 실행 가능한 Hugging Face Transformers 기반 한국어 BERT 감정 분석 예제입니다.

포함 내용:
1. 네이버 영화 리뷰 데이터셋 NSMC 로드
2. 한국어 BERT 모델 사용
3. `AutoTokenizer`로 전처리
4. `AutoModelForSequenceClassification` 사용
5. `Trainer` API로 학습 수행
6. accuracy 평가
7. 샘플 문장 예측 결과 출력

참고: Kaggle CPU 환경에서도 빠르게 실행되도록 NSMC 데이터셋 일부만 사용합니다. 성능을 높이려면 train/eval 샘플 수와 epoch 수를 늘리면 됩니다.

In [ ]:
# Kaggle Notebook에서 필요한 라이브러리 설치
# transformers: 한국어 BERT 모델과 Trainer API
# datasets: NSMC TSV 파일 로드 및 Dataset 변환
# accelerate: Trainer 실행에 필요
!pip install -q -U "transformers==4.48.3" datasets accelerate

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import numpy as np
import torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 1. NSMC 데이터셋 로드

NSMC는 네이버 영화 리뷰를 부정/긍정으로 분류하는 한국어 감정 분석 데이터셋입니다. label은 `0: 부정`, `1: 긍정`입니다.

최신 `datasets`에서는 script 기반 `load_dataset("nsmc")`가 막힐 수 있으므로, NSMC 원본 TSV 파일을 URL에서 직접 읽어 `datasets`의 CSV 로더로 불러옵니다.

In [ ]:
# NSMC 원본 TSV 파일을 URL에서 직접 로드합니다.
# load_dataset("nsmc")는 최신 datasets 버전에서 script 기반 데이터셋 제한으로 실패할 수 있습니다.
data_files = {
    "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
}

raw_datasets = load_dataset(
    "csv",
    data_files=data_files,
    delimiter="\t",
)

# document가 비어 있는 일부 행을 제거합니다.
raw_datasets = raw_datasets.filter(lambda example: example["document"] is not None)

# Kaggle에서 빠르게 실행되도록 일부 샘플만 사용합니다.
# 실제 성능을 높이려면 train/eval 샘플 수를 늘리세요.
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(1000))
eval_dataset = raw_datasets["test"].shuffle(seed=42).select(range(300))

print(train_dataset)
print(eval_dataset)
print("\nSample:")
print(train_dataset[0])

## 2. AutoTokenizer로 전처리

한국어 BERT 모델인 `klue/bert-base`의 토크나이저를 사용해 리뷰 문장을 모델 입력 형태로 변환합니다.

In [ ]:
# 한국어 BERT 모델과 토크나이저 이름입니다.
# klue/bert-base는 한국어 자연어 이해 벤치마크 KLUE 기반의 BERT 모델입니다.
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# NSMC의 문장 컬럼은 document입니다.
# truncation=True로 긴 문장을 max_length 안에서 자릅니다.
def tokenize_function(batch):
    return tokenizer(batch["document"], truncation=True, max_length=128)

# train/eval 데이터셋에 토크나이징을 적용합니다.
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 배치마다 동적으로 padding을 적용합니다.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_train[0].keys())

## 3. AutoModelForSequenceClassification 로드

`AutoModelForSequenceClassification`으로 한국어 BERT를 이진 감정 분류 모델로 불러옵니다.

In [ ]:
# NSMC는 부정/긍정 이진 분류입니다.
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

## 4. Accuracy 평가 함수 정의

`Trainer` 평가 단계에서 사용할 accuracy 계산 함수를 정의합니다.

In [ ]:
# 모델 출력 logits에서 가장 큰 값을 가진 class를 예측 label로 사용합니다.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = np.mean(predictions == labels)
    return {"accuracy": accuracy}

## 5. Trainer API로 학습

`TrainingArguments`와 `Trainer`를 설정한 뒤 NSMC subset으로 한국어 BERT를 fine-tuning합니다.

In [ ]:
# Kaggle에서 빠르게 실행되도록 epoch과 batch size를 작게 설정합니다.
training_args = TrainingArguments(
    output_dir="./bert-nsmc-results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
)

# Trainer가 학습 루프, 평가 루프, 배치 구성, metric 계산을 처리합니다.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 한국어 BERT 감정 분석 모델 학습을 수행합니다.
trainer.train()

## 6. 평가 Accuracy 확인

학습된 모델을 NSMC 평가 subset으로 평가하고 accuracy를 출력합니다.

In [ ]:
# 평가 데이터셋에 대한 loss와 accuracy를 계산합니다.
eval_results = trainer.evaluate()

print("Evaluation results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## 7. 샘플 문장 예측 결과 출력

학습된 한국어 BERT 모델로 새로운 한국어 영화 리뷰 문장을 긍정/부정으로 분류합니다.

In [ ]:
# 예측할 한국어 영화 리뷰 샘플입니다.
sample_texts = [
    "정말 재미있고 감동적인 영화였습니다. 배우들의 연기도 훌륭했어요.",
    "시간이 아까울 정도로 지루하고 실망스러운 영화였습니다.",
    "초반은 괜찮았지만 후반으로 갈수록 전개가 아쉬웠습니다.",
]

# 샘플 문장을 모델 입력 형태로 변환합니다.
inputs = tokenizer(
    sample_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
)

# 입력 tensor를 모델이 위치한 device로 이동합니다.
device = trainer.model.device
inputs = {key: value.to(device) for key, value in inputs.items()}

# 예측 시에는 gradient 계산이 필요 없으므로 torch.no_grad()를 사용합니다.
trainer.model.eval()
with torch.no_grad():
    outputs = trainer.model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_labels = torch.argmax(probabilities, dim=-1)

# 예측 label과 confidence를 출력합니다.
for text, label_id, probs in zip(sample_texts, predicted_labels, probabilities):
    label_id = label_id.item()
    confidence = probs[label_id].item()
    korean_label = "긍정" if label_id == 1 else "부정"
    print("문장:", text)
    print("예측:", korean_label, f"({id2label[label_id]})")
    print("신뢰도:", round(confidence, 4))
    print("-" * 100)